# Phase 4: Gap Detection

Identify what's **missing** in the video dataset using two complementary approaches:

- **Mode A (Structural):** Flag sparse clusters, isolated clusters, and compute UMAP space coverage
- **Mode B (Category-driven):** Embed expected categories via Marengo text API, compare to cluster centroids, flag gaps

**Prerequisites:** Run `01_embeddings.py`, `02_clustering.py`, and `03_cluster_descriptions.py` first so the dataset has `embedding`, `cluster_id`, `centroid_distance`, `umap_x`, `umap_y`, and `cluster_label` fields.

## 1. Setup & Imports

In [1]:
import os
import time

import numpy as np
import fiftyone as fo
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from sklearn.preprocessing import normalize
from twelvelabs import TwelveLabs, TextInputRequest

# Validate API key
api_key = os.environ.get("TWELVELABS_API_KEY")
assert api_key, "Set TWELVELABS_API_KEY environment variable first"
client = TwelveLabs(api_key=api_key)
print("Twelve Labs client ready")

# Configuration
SPARSE_THRESHOLD = 3
GAP_SIMILARITY_THRESHOLD = 0.10  # calibrated for cross-modal text-video similarity
COVERAGE_GRID_SIZE = 10
RATE_LIMIT_WAIT = 30
EXPECTED_CATEGORIES = [
    "factory floor with industrial machinery",
    "worker in safety helmet operating equipment",
    "person falling",
    "forklift moving",
    "emergency evacuation",
    "fire or smoke in building",
    "machine malfunction",
]

/Users/rishimule/Desktop/GitHub/video-content-gap-analyzer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Twelve Labs client ready


## 2. Load Dataset & Validate Fields

In [2]:
dataset = fo.load_dataset("Voxel51/Safe_and_Unsafe_Behaviours")
print(f"Dataset: {dataset.name} ({len(dataset)} samples)")

# Verify prerequisite fields
sample = dataset.first()
for field in ["embedding", "cluster_id", "umap_x", "umap_y", "cluster_label"]:
    assert sample[field] is not None, f"Run prerequisite scripts first (missing {field})"
print("All prerequisite fields verified")

Dataset: Voxel51/Safe_and_Unsafe_Behaviours (200 samples)
All prerequisite fields verified


## 3. Extract Cluster Data & Compute Centroids

Collect all samples with embeddings and cluster assignments, then recompute L2-normalized centroids from the sample embeddings.

In [3]:
# Extract samples with all required fields
samples_list = []
embeddings_list = []
cluster_ids_list = []
umap_list = []
cluster_labels_map = {}

for sample in dataset:
    try:
        emb = sample["embedding"]
        cid = sample["cluster_id"]
        ux = sample["umap_x"]
        uy = sample["umap_y"]
    except (KeyError, AttributeError):
        continue
    if emb is None or cid is None or ux is None or uy is None:
        continue

    samples_list.append(sample)
    embeddings_list.append(emb)
    cluster_ids_list.append(cid)
    umap_list.append([ux, uy])

    if cid not in cluster_labels_map:
        try:
            label = sample["cluster_label"]
            cluster_labels_map[cid] = label if label else f"Cluster {cid}"
        except (KeyError, AttributeError):
            cluster_labels_map[cid] = f"Cluster {cid}"

embeddings = np.array(embeddings_list)
cluster_ids = np.array(cluster_ids_list, dtype=int)
umap_coords = np.array(umap_list)

n_samples = len(samples_list)
n_clusters = len(cluster_labels_map)
print(f"Extracted {n_samples} samples across {n_clusters} clusters")
print(f"Embedding matrix: {embeddings.shape}")

# L2 normalize and compute centroids
embeddings_norm = normalize(embeddings, norm="l2")

unique_ids = np.sort(np.unique(cluster_ids))
centroids = []
for cid in unique_ids:
    mask = cluster_ids == cid
    centroids.append(embeddings_norm[mask].mean(axis=0))
centroids = normalize(np.array(centroids), norm="l2")

print(f"Centroid matrix: {centroids.shape}")
for cid in unique_ids:
    cnt = (cluster_ids == cid).sum()
    print(f"  Cluster {cid}: {cnt} samples")

Extracted 30 samples across 2 clusters
Embedding matrix: (30, 512)
Centroid matrix: (2, 512)
  Cluster 0: 14 samples
  Cluster 1: 16 samples


## 4. Mode A: Structural Gap Analysis

Detect gaps without any user input:
- **Sparse clusters:** fewer than 3 samples
- **Isolated clusters:** centroid far from all others (requires >= 3 clusters)
- **UMAP coverage:** fraction of 10x10 bounding box grid occupied by samples

In [4]:
# --- Sparse cluster detection ---
unique, counts = np.unique(cluster_ids, return_counts=True)
sparse_clusters = []
for cid, cnt in zip(unique, counts):
    if cnt < SPARSE_THRESHOLD:
        sparse_clusters.append({
            "cluster_id": int(cid),
            "label": cluster_labels_map.get(int(cid), f"Cluster {cid}"),
            "count": int(cnt),
        })

print(f"Sparse clusters (< {SPARSE_THRESHOLD} samples): {len(sparse_clusters)}")
if sparse_clusters:
    for sc in sparse_clusters:
        print(f"  Cluster {sc['cluster_id']} ({sc['count']} samples): \"{sc['label'][:60]}...\"")
else:
    print("  None — all clusters have sufficient samples")

# --- Isolated cluster detection ---
if n_clusters >= 3:
    dist_matrix = cosine_distances(centroids, centroids)
    mean_dists = []
    for i in range(n_clusters):
        others = [dist_matrix[i, j] for j in range(n_clusters) if j != i]
        mean_dists.append(np.mean(others))
    mean_dists = np.array(mean_dists)
    iso_threshold = mean_dists.mean() + 1.5 * mean_dists.std()

    isolated_clusters = []
    for i, cid in enumerate(unique_ids):
        if mean_dists[i] > iso_threshold:
            isolated_clusters.append({
                "cluster_id": int(cid),
                "label": cluster_labels_map.get(int(cid), f"Cluster {cid}"),
                "mean_inter_distance": round(float(mean_dists[i]), 4),
            })
    print(f"\nIsolated clusters: {len(isolated_clusters)}")
else:
    isolated_clusters = []
    print(f"\nIsolated cluster detection: skipped (need >= 3 clusters, have {n_clusters})")

# --- UMAP grid coverage ---
x_min, x_max = umap_coords[:, 0].min(), umap_coords[:, 0].max()
y_min, y_max = umap_coords[:, 1].min(), umap_coords[:, 1].max()
x_range = x_max - x_min
y_range = y_max - y_min

if x_range > 0 and y_range > 0:
    x_min -= 0.01 * x_range
    x_max += 0.01 * x_range
    y_min -= 0.01 * y_range
    y_max += 0.01 * y_range
    cell_w = (x_max - x_min) / COVERAGE_GRID_SIZE
    cell_h = (y_max - y_min) / COVERAGE_GRID_SIZE

    occupied = set()
    for x, y in umap_coords:
        cx = min(int((x - x_min) / cell_w), COVERAGE_GRID_SIZE - 1)
        cy = min(int((y - y_min) / cell_h), COVERAGE_GRID_SIZE - 1)
        occupied.add((cx, cy))
    umap_coverage = len(occupied) / (COVERAGE_GRID_SIZE ** 2)
else:
    umap_coverage = 0.0

print(f"\nUMAP grid coverage: {umap_coverage * 100:.1f}% "
      f"({len(occupied)}/{COVERAGE_GRID_SIZE ** 2} cells occupied)")

Sparse clusters (< 3 samples): 0
  None — all clusters have sufficient samples

Isolated cluster detection: skipped (need >= 3 clusters, have 2)

UMAP grid coverage: 21.0% (21/100 cells occupied)


## 5. Mode B: Category-Driven Gap Analysis

Embed expected category strings via Marengo text API, then compute cosine similarity against each cluster centroid. Categories with max similarity below 0.3 are flagged as **gaps** — the dataset has no matching content.

In [5]:
# Embed each expected category via Marengo text API
category_embeddings = {}

for i, category in enumerate(EXPECTED_CATEGORIES, start=1):
    print(f"[{i}/{len(EXPECTED_CATEGORIES)}] Embedding: \"{category}\" ... ", end="", flush=True)
    start = time.time()

    for attempt in range(2):
        try:
            response = client.embed.v_2.create(
                input_type="text",
                model_name="marengo3.0",
                text=TextInputRequest(input_text=category),
            )
            embedding = np.array(response.data[0].embedding)
            embedding = normalize(embedding.reshape(1, -1), norm="l2")[0]
            category_embeddings[category] = embedding

            elapsed = time.time() - start
            print(f"{len(embedding)}-d ({elapsed:.1f}s)")
            break
        except Exception as e:
            elapsed = time.time() - start
            error_str = str(e).lower()
            if "429" in str(e) or "rate" in error_str or "too many" in error_str:
                if attempt == 0:
                    print(f"rate limited, waiting {RATE_LIMIT_WAIT}s ... ", end="", flush=True)
                    time.sleep(RATE_LIMIT_WAIT)
                    continue
            print(f"FAILED ({elapsed:.1f}s): {e}")
            break

print(f"\n{len(category_embeddings)}/{len(EXPECTED_CATEGORIES)} categories embedded")

[1/7] Embedding: "factory floor with industrial machinery" ... 512-d (0.5s)
[2/7] Embedding: "worker in safety helmet operating equipment" ... 512-d (0.3s)
[3/7] Embedding: "person falling" ... 512-d (0.2s)
[4/7] Embedding: "forklift moving" ... 512-d (0.2s)
[5/7] Embedding: "emergency evacuation" ... 512-d (0.2s)
[6/7] Embedding: "fire or smoke in building" ... 512-d (0.2s)
[7/7] Embedding: "machine malfunction" ... 512-d (0.2s)

7/7 categories embedded


In [6]:
# Compute cosine similarity between each category and ALL individual sample embeddings
categories = list(category_embeddings.keys())
cat_matrix = np.array([category_embeddings[c] for c in categories])
sim_matrix = cosine_similarity(cat_matrix, embeddings_norm)

category_results = []
for i, category in enumerate(categories):
    max_sim = float(sim_matrix[i].max())
    closest_sample_idx = int(sim_matrix[i].argmax())
    closest_cid = int(cluster_ids[closest_sample_idx])
    closest_label = cluster_labels_map.get(closest_cid, f"Cluster {closest_cid}")

    is_gap = max_sim < GAP_SIMILARITY_THRESHOLD
    category_results.append({
        "category": category,
        "closest_cluster": closest_label,
        "closest_cluster_id": closest_cid,
        "similarity": round(max_sim, 4),
        "is_gap": is_gap,
    })

# Display results
n_gaps = sum(1 for cr in category_results if cr["is_gap"])
print(f"Categories checked: {len(category_results)}")
print(f"Gaps found: {n_gaps}\n")

max_cat_len = max(len(cr["category"]) for cr in category_results)
for cr in category_results:
    status = "GAP" if cr["is_gap"] else "COVERED"
    label_short = cr["closest_cluster"][:50] + "..." if len(cr["closest_cluster"]) > 50 else cr["closest_cluster"]
    print(f"  {cr['category']:<{max_cat_len}} -> {status:>7} "
          f"(sim: {cr['similarity']:.2f}, nearest: \"{label_short}\")")

Categories checked: 7
Gaps found: 5

  factory floor with industrial machinery     -> COVERED (sim: 0.16, nearest: "The video captures a static shot of a bustling fac...")
  worker in safety helmet operating equipment -> COVERED (sim: 0.11, nearest: "The video captures a factory environment where a w...")
  person falling                              ->     GAP (sim: 0.03, nearest: "The video captures a factory environment where a w...")
  forklift moving                             ->     GAP (sim: 0.09, nearest: "The video captures a factory environment where a w...")
  emergency evacuation                        ->     GAP (sim: 0.03, nearest: "The video captures a factory environment where a w...")
  fire or smoke in building                   ->     GAP (sim: 0.05, nearest: "The video captures a factory environment where a w...")
  machine malfunction                         ->     GAP (sim: 0.08, nearest: "The video captures a static shot of a bustling fac...")


## 6. Tag Samples & Store Gap Report

Tag samples in sparse clusters, compute the combined coverage score, and store the gap report in `dataset.info["gap_report"]`.

In [7]:
# Tag samples in sparse clusters
sparse_ids = {sc["cluster_id"] for sc in sparse_clusters}
tagged = 0
for sample in dataset:
    try:
        cid = sample["cluster_id"]
    except (KeyError, AttributeError):
        continue
    if cid is not None and cid in sparse_ids:
        if "sparse_cluster" not in sample.tags:
            sample.tags.append("sparse_cluster")
            sample.save()
            tagged += 1
print(f"Tagged {tagged} samples with 'sparse_cluster'")

# Compute combined coverage score
if category_results:
    n_covered = sum(1 for cr in category_results if not cr["is_gap"])
    category_coverage = n_covered / len(category_results)
    combined_score = 0.5 * umap_coverage + 0.5 * category_coverage
else:
    category_coverage = 0.0
    combined_score = umap_coverage

# Build and store gap report
gap_report = {
    "sparse_clusters": sparse_clusters,
    "category_gaps": [
        {
            "category": cr["category"],
            "closest_cluster": cr["closest_cluster"],
            "similarity": cr["similarity"],
        }
        for cr in category_results if cr["is_gap"]
    ],
    "coverage_score": round(float(combined_score), 4),
}

dataset.info["gap_report"] = gap_report
dataset.save()
print(f"\nGap report stored in dataset.info[\"gap_report\"]")
print(f"Coverage score: {combined_score:.2f}")

Tagged 0 samples with 'sparse_cluster'

Gap report stored in dataset.info["gap_report"]
Coverage score: 0.25


## 7. Summary Report

In [8]:
print("=" * 60)
print("GAP DETECTION REPORT")
print("=" * 60)

print(f"\nSTRUCTURAL ANALYSIS (Mode A)")
print(f"  Clusters: {n_clusters} total, {len(sparse_clusters)} sparse, "
      f"{len(isolated_clusters)} isolated")
print(f"  UMAP coverage: {umap_coverage * 100:.1f}%")

if sparse_clusters:
    print(f"\n  Sparse clusters (< {SPARSE_THRESHOLD} samples):")
    for sc in sparse_clusters:
        print(f"    Cluster {sc['cluster_id']} ({sc['count']} samples)")
else:
    print(f"  No sparse clusters")

if n_clusters < 3:
    print(f"  Isolation detection: skipped (need >= 3 clusters)")

if category_results:
    n_gaps = sum(1 for cr in category_results if cr["is_gap"])
    print(f"\nCATEGORY GAP ANALYSIS (Mode B)")
    print(f"  Categories: {len(category_results)}, Gaps: {n_gaps}")

    max_cat_len = max(len(cr["category"]) for cr in category_results)
    for cr in category_results:
        status = "GAP" if cr["is_gap"] else "COVERED"
        print(f"  {cr['category']:<{max_cat_len}} -> {status:>7} (sim: {cr['similarity']:.2f})")

print(f"\nCOVERAGE SCORE: {combined_score:.2f}", end="")
if category_results:
    print(f"  (structural: {umap_coverage:.2f}, category: {category_coverage:.2f})")
else:
    print()

print(f"\nDataset: {dataset.name}")
print(f"Report: dataset.info[\"gap_report\"]")

GAP DETECTION REPORT

STRUCTURAL ANALYSIS (Mode A)
  Clusters: 2 total, 0 sparse, 0 isolated
  UMAP coverage: 21.0%
  No sparse clusters
  Isolation detection: skipped (need >= 3 clusters)

CATEGORY GAP ANALYSIS (Mode B)
  Categories: 7, Gaps: 5
  factory floor with industrial machinery     -> COVERED (sim: 0.16)
  worker in safety helmet operating equipment -> COVERED (sim: 0.11)
  person falling                              ->     GAP (sim: 0.03)
  forklift moving                             ->     GAP (sim: 0.09)
  emergency evacuation                        ->     GAP (sim: 0.03)
  fire or smoke in building                   ->     GAP (sim: 0.05)
  machine malfunction                         ->     GAP (sim: 0.08)

COVERAGE SCORE: 0.25  (structural: 0.21, category: 0.29)

Dataset: Voxel51/Safe_and_Unsafe_Behaviours
Report: dataset.info["gap_report"]


## 8. Launch FiftyOne App

Browse the dataset with gap analysis results. Samples in sparse clusters are tagged with `sparse_cluster`.

In [9]:
session = fo.launch_app(dataset)

Connected to FiftyOne on port 5151 at localhost.
If you are not connecting to a remote session, you may need to start a new session and specify a port
